# Notebook 15: 101 Formulaic Alphas — Engine, Diagnostics & Selection

Walks through the full WorldQuant 101 alpha pipeline for the 10-stock universe:

1. **Alpha engine**: compute all 101 formulas on the OHLCV panel
2. **Diagnostics**: per-alpha NaN rate, stationarity (ADF), autocorrelation, skew
3. **Exclusion rules**: drop alphas with >40% NaN, zero std, or any Inf
4. **Redundancy pruning**: remove one from each pair with |corr| > 0.85 (keep more stationary)
5. **Budget**: cap to top-33 by ADF stationarity

**Universe**: 10 stocks (AAPL, AMZN, BAC, GOOGL, JNJ, JPM, MSFT, NVDA, UNH, XOM)  
**Period**: 2005-01-03 to 2025-04-30 (≈51,138 panel rows)  
**Output**: `configs/selected_alphas.json` + `panel_alpha_features_pruned.parquet`

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../..'))
from src.alphas.diagnostics import run_full_diagnostics

os.makedirs('../../reports/figures', exist_ok=True)

with open('../../configs/selected_alphas.json') as f:
    sel_cfg = json.load(f)

print('Selected alphas:', sel_cfg['n_alphas'])
print('Excluded (NaN>40%):', len(sel_cfg['excluded_nan40']))
print('Excluded (constant):', len(sel_cfg['excluded_constant']))
print('Excluded (inf)     :', len(sel_cfg['excluded_inf']))
print('Pruned (redundant) :', len(sel_cfg['pruned_redundant']))

## 1. Load alpha panel

In [ ]:
alpha_panel = pd.read_parquet('../../data/processed/panel_alpha_features.parquet')
alpha_pruned = pd.read_parquet('../../data/processed/panel_alpha_features_pruned.parquet')

print(f'Full panel  : {alpha_panel.shape}  (rows x 101 alphas)')
print(f'Pruned panel: {alpha_pruned.shape}  (rows x {len(alpha_pruned.columns)} alphas)')
print(f'Date range  : {alpha_panel.index.get_level_values("Date").min().date()} -> '
      f'{alpha_panel.index.get_level_values("Date").max().date()}')
print(f'Tickers     : {sorted(alpha_panel.index.get_level_values("ticker").unique())}')

## 2. NaN / Inf summary across all 101 alphas

In [ ]:
nan_pcts = alpha_panel.isnull().mean() * 100
stds     = alpha_panel.std()

print(f'Alphas < 20% NaN   : {(nan_pcts < 20).sum()}')
print(f'Alphas 20-40% NaN  : {((nan_pcts >= 20) & (nan_pcts < 40)).sum()}')
print(f'Alphas 40-60% NaN  : {((nan_pcts >= 40) & (nan_pcts < 60)).sum()}')
print(f'Alphas > 60% NaN   : {(nan_pcts >= 60).sum()}')
print(f'Constant alphas    : {(stds < 1e-8).sum()}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(nan_pcts)), nan_pcts.values, color='steelblue', alpha=0.7)
ax.axhline(40, color='red', linestyle='--', label='40% exclusion threshold')
ax.set_xlabel('Alpha index')
ax.set_ylabel('NaN %')
ax.set_title('NaN Rate per Alpha (all 101)')
ax.legend()
plt.tight_layout()
plt.savefig('../../reports/figures/pooled/9_alpha_nan_rates.png', dpi=100)
plt.show()
print('Top 10 NaN rates:')
print(nan_pcts.nlargest(10).round(1).to_string())

## 3. ADF stationarity diagnostics

In [ ]:
diag = pd.read_parquet('../../data/processed/alpha_diagnostics.parquet')
print(f'Diagnostics shape : {diag.shape}')
print(f'Columns           : {list(diag.columns)}')
print()
print(f'Stationary (ADF p<0.05) : {(diag["adf_pval_median"] < 0.05).sum()} / {len(diag)}')
print(f'Median ADF p-value      : {diag["adf_pval_median"].median():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(diag['adf_pval_median'].clip(0, 1), bins=30, color='steelblue', edgecolor='k', alpha=0.7)
axes[0].axvline(0.05, color='red', linestyle='--', label='p=0.05')
axes[0].set_xlabel('Median ADF p-value')
axes[0].set_ylabel('Count')
axes[0].set_title('ADF p-value Distribution (all alphas)')
axes[0].legend()

axes[1].scatter(diag['adf_pval_median'], diag.get('autocorr_lag1', pd.Series(dtype=float)),
                alpha=0.5, s=20, color='steelblue')
axes[1].axvline(0.05, color='red', linestyle='--')
axes[1].set_xlabel('Median ADF p-value')
axes[1].set_ylabel('Lag-1 Autocorrelation')
axes[1].set_title('Stationarity vs Autocorrelation')

plt.tight_layout()
plt.savefig('../../reports/figures/pooled/9_alpha_adf_diagnostics.png', dpi=100)
plt.show()

## 4. Exclusion → Pruning → Budget pipeline

In [ ]:
print('=== Alpha Selection Pipeline ===' )
print(f'  Start              : 101 alphas')
print(f'  After exclusion    : {101 - len(sel_cfg["excluded_nan40"]) - len(sel_cfg["excluded_constant"]) - len(sel_cfg["excluded_inf"])} alphas')
print(f'    Excluded NaN>40% : {sel_cfg["excluded_nan40"]}')
print(f'    Excluded constant: {sel_cfg["excluded_constant"]}')
print(f'    Excluded inf     : {sel_cfg["excluded_inf"]}')
print(f'  Pruned redundant   : {sel_cfg["pruned_redundant"]}')
print(f'  Final selected ({sel_cfg["n_alphas"]}): {sel_cfg["selected_alphas"]}')

## 5. Selected alpha correlation heatmap (NVDA slice)

In [ ]:
selected = sel_cfg['selected_alphas']
nvda_slice = alpha_panel[selected].xs('NVDA', level='ticker')
corr = nvda_slice.corr()

n = len(selected)
fig, ax = plt.subplots(figsize=(max(10, n * 0.4), max(8, n * 0.4)))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.index, fontsize=7)
plt.colorbar(im, ax=ax)
ax.set_title(f'Selected Alpha Correlation (NVDA, n={n})')
plt.tight_layout()
plt.savefig('../../reports/figures/pooled/9_selected_alpha_correlation.png', dpi=100)
plt.show()

corr_arr = corr.to_numpy().copy()
np.fill_diagonal(corr_arr, 0)
print(f'Max |corr| in selected set: {np.nanmax(np.abs(corr_arr)):.4f}  (should be <= 0.85)')